# Chat function with GROK


This will only work with the hidden API_KEY function in google colab... my hidden XAI key is called: 'x_ai_API_KEY', so that is how we have referenced to the API_key in the code

In [ ]:
!pip install --upgrade openai PyPDF2 requests

In [ ]:
from google.colab import userdata
import os
x_ai_API_KEY = userdata.get('x_ai_API_KEY')
os.environ["XAI_API_KEY"] = x_ai_API_KEY

In [ ]:
import os
import json
import requests
import PyPDF2
import io
from typing import List, Dict, Any
from openai import OpenAI

In [ ]:
class PDFChatAssistant:
    def __init__(self, pdf_path: str, x_ai_api_key: str):
        """
        Initialize the PDF Chat Assistant with Grok

        :param pdf_path: Path to the PDF file
        :param x_ai_api_key: X.AI API key for Grok
        """
        self.pdf_path = pdf_path

        # Extract text from PDF
        self.pdf_text = self._extract_pdf_text()

        # Define functions for the assistant
        self.functions = [
            {
                "name": "search_pdf_text",
                "description": "Search through the PDF text for relevant information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "Search query to find relevant text in the PDF"
                        },
                        "max_results": {
                            "type": "integer",
                            "description": "Maximum number of text snippets to return",
                            "default": 5
                        }
                    },
                    "required": ["query"]
                }
            }
        ]

        # Initialize Grok
        self.client = OpenAI(
            api_key=x_ai_api_key,
            base_url="https://api.x.ai/v1"
        )

        # Initialize conversation history
        self.messages = [
            {
                "role": "system",
                "content": f"""You are a helpful PDF assistant powered by Grok. You have access to the full text of a PDF document.
                Your goal is to assist the user by providing accurate and contextual information from the document.
                When you need specific information, use the search_pdf_text function to find relevant passages.

                PDF Content:
                {self.pdf_text}

                Available PDF Document: The document has been preprocessed and is ready for searching.
                Total document length: {len(self.pdf_text)} characters

                Guidelines:
                1. Always use the search_pdf_text function to find relevant information
                2. Provide clear and concise answers
                3. If you cannot find relevant information, state 'I cannot answer this question.'
                4. Do not use any information outside of the provided text
                """
            }
        ]

    def _extract_pdf_text(self) -> str:
        """
        Extract text from PDF file

        :return: Extracted text from PDF
        """
        with open(self.pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
        return text

    def search_pdf_text(self, query: str, max_results: int = 3) -> List[str]:
        """
        Search through PDF text and return relevant snippets

        :param query: Search query
        :param max_results: Maximum number of results to return
        :return: List of relevant text snippets
        """
        results = []
        lines = self.pdf_text.split('\n')

        for line in lines:
            if query.lower() in line.lower():
                results.append(line.strip())
                if len(results) == max_results:
                    break

        return results

    def chat(self, user_message: str) -> str:
        """
        Chat with the PDF assistant

        :param user_message: User's input message
        :return: Assistant's response
        """
        # Add user message to conversation history
        self.messages.append({"role": "user", "content": user_message})

        # Prepare tools for function calling
        tools = [{"type": "function", "function": f} for f in self.functions]

        # Create chat completion with Grok
        response = self.client.chat.completions.create(
            model="grok-beta",
            messages=self.messages,
            tools=tools,
            tool_choice="auto"
        )

        # Check if function call is needed
        if response.choices[0].message.tool_calls:
            # Process tool calls
            for tool_call in response.choices[0].message.tool_calls:
                function_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)

                if function_name == "search_pdf_text":
                    search_results = self.search_pdf_text(
                        query=arguments.get('query'),
                        max_results=arguments.get('max_results', 3)
                    )

                    # Add function result to messages
                    self.messages.append({
                        "role": "tool",
                        "content": json.dumps(search_results),
                        "tool_call_id": tool_call.id
                    })

                    # Get final response with search results
                    final_response = self.client.chat.completions.create(
                        model="grok-beta",
                        messages=self.messages
                    )

                    assistant_response = final_response.choices[0].message.content
                    self.messages.append({"role": "assistant", "content": assistant_response})

                    return assistant_response

        # If no function call needed, get direct response
        assistant_response = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_response})

        return assistant_response

In [ ]:
# Example usage in Google Colab
def main():
    from google.colab import userdata
    import os

    # Retrieve the API key securely
    x_ai_API_KEY = userdata.get('x_ai_API_KEY')

    # You'll need to replace this with your actual PDF path
    PDF_PATH = "/content/target.pdf"

    # Initialize PDF Chat Assistant
    pdf_chat = PDFChatAssistant(PDF_PATH, x_ai_API_KEY)

    # Interactive chat loop
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            break

        response = pdf_chat.chat(user_input)
        print("Assistant:", response)

if __name__ == "__main__":
    main()

You: What is the main theory?
Assistant: The main theory discussed in the document is **political consumerism** as a form of political participation. It explores how consumer choices based on political or ethical considerations can be seen as a significant form of political activism, particularly in the context of modern democratic societies where traditional forms of political engagement might be declining or evolving.
You: Can you list the chapters or main sections of the document?
Assistant: The document 'Politics in the Supermarket: Political Consumerism as a Form of Political Participation' by Dietlind Stolle, Marc Hooghe, and Michele Micheletti is structured with the following main sections:

1. **Abstract** - Provides an overview of the study, discussing the significance of political consumerism and the methodology used.

2. **Introduction** - Introduces the concept of political consumerism, using the example of the 2003 anti-French sentiment in the U.S. to illustrate its politi

KeyboardInterrupt: Interrupted by user

**Recommendations for questions to check the discussion funtion:**
* What is the main theory?
  - The purpose of this is to check if the chatfunction reads and ansvers based on the provided text
* Can you list the chapters or main sections of the document?
  - Test of basic comprehension and relative evaluation
* What is the author’s purpose or perspective in this document?
  - Understand chat ability to evaluate context
* How many fingers am I holding up?
  - The purpose is to check the Chatbots ability to tell if the desired information is in the provided PDF or not
* What is the purpose of life based in the paper?
  - This tests the flexibility of the model... if it is able to speculate with abstract constructs in relation to a provided text